# Iterators, Generators, Decorators and Context Managers

Notebook for **Chapter 7** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Iterators

In [ ]:
nums = [10, 20, 30]
it = iter(nums)
print(type(it).__name__, next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration: exhausted")
print(iter(it) is it, iter(nums) is nums)

it = iter(nums)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    print("got", x)

it = iter([1, 2, 3])
print(list(it), list(it))
print(next(iter([]), "default"))

list_iterator 10 20 30
StopIteration: exhausted
True False
got 10
got 20
got 30
[1, 2, 3] []
default


In [ ]:
it = iter([1, 2, 3])
print(sum(it))   # 6
print(sum(it))   # 0, silently

In [ ]:
class Countdown:
    def __init__(self, start):
        self.n = start
    def __iter__(self):
        return self
    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

print(list(Countdown(4)))
print(sum(Countdown(4)), max(Countdown(3)))
d = {"a": 1, "b": 2}
print(list(iter(d)), list(d.items()))
print(list(enumerate("ab", start=1)), list(zip("ab", [1, 2, 3])))

[4, 3, 2, 1]
10 3
['a', 'b'] [('a', 1), ('b', 2)]
[(1, 'a'), (2, 'b')] [('a', 1), ('b', 2)]


## 7.2 Generators

In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

g = squares(4)
print(type(g).__name__, next(g), next(g), list(g))

def trace():
    print("start"); yield 1
    print("middle"); yield 2
    print("end")

t = trace()
print("created, nothing printed yet")
print(next(t)); print(next(t))
try:
    next(t)
except StopIteration:
    print("done")

generator 0 1 [4, 9]
created, nothing printed yet
start
1
middle
2
end
done


In [ ]:
big_list = [i*i for i in range(1_000_000)]   # all 1 million numbers in memory
big_gen  = (i*i for i in range(1_000_000))   # nothing computed yet

In [ ]:
import sys
big_list = [i*i for i in range(1_000_000)]
big_gen = (i*i for i in range(1_000_000))
print(sys.getsizeof(big_list), sys.getsizeof(big_gen))
print(sum(big_gen) == sum(big_list))

8448728 208
True


In [ ]:
lines = ["OK north A1 120", "LATE south B2 900", "OK east C3 340",
         "bad line", "OK west D4 95"]

def parse(ls):
    for l in ls:
        parts = l.split()
        if len(parts) == 4:
            yield parts[0], parts[1], parts[2], int(parts[3])

def only_ok(rs):
    return (r for r in rs if r[0] == "OK")

def times(rs):
    return (r[3] for r in rs)

ts = list(times(only_ok(parse(lines))))
print(ts, sum(ts) / len(ts))

[120, 340, 95] 185.0


In [ ]:
from itertools import islice

def inner():
    yield from [1, 2]
    yield from "ab"
print(list(inner()))

def chain_flat(nested):
    for sub in nested:
        yield from sub
print(list(chain_flat([[1, 2], [3], [4, 5]])))

def fib():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b
print(list(islice(fib(), 10)))

def running_avg():
    total = n = 0
    avg = None
    while True:
        x = yield avg
        total += x
        n += 1
        avg = total / n
r = running_avg(); next(r)
print(r.send(10), r.send(20), r.send(60))

def gen_read(lines, size):
    batch = []
    for l in lines:
        batch.append(l)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch
print(list(gen_read(range(7), 3)))

[1, 2, 'a', 'b']
[1, 2, 3, 4, 5]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
10.0 15.0 30.0
[[0, 1, 2], [3, 4, 5], [6]]


## 7.3 Decorators

In [ ]:
@shout
def add(a, b): ...
# same as:
def add(a, b): ...
add = shout(add)

In [ ]:
import functools

def shout(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper

@shout
def add(a, b):
    "Add two numbers."
    return a + b
print(add(2, 3), add.__name__, add.__doc__)

def shout2(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@shout2
def add2(a, b):
    "Add two numbers."
    return a + b
print(add2.__name__, add2.__doc__)

calling add
5 wrapper None
add2 Add two numbers.


In [ ]:
import functools, time

def timed(func):
    @functools.wraps(func)
    def wrapper(*a, **k):
        t0 = time.perf_counter()
        try:
            return func(*a, **k)
        finally:
            ms = (time.perf_counter() - t0) * 1000
            print(f"{func.__name__} took >=5ms: {ms >= 5}")
    return wrapper

@timed
def slow():
    time.sleep(0.01)
    return "done"
print(slow())

def repeat(times):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            return [func(*a, **k) for _ in range(times)]
        return wrapper
    return deco

@repeat(3)
def hi(name):
    return f"hi {name}"
print(hi("Ravi"))

slow took >=5ms: True
done
['hi Ravi', 'hi Ravi', 'hi Ravi']


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
@repeat          # should be @repeat(3)
def bad(): pass
bad()

TypeError: repeat.<locals>.deco() missing 1 required positional argument: 'func'


In [ ]:
import functools

def retry(max_tries=3, exceptions=(Exception,)):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            for attempt in range(1, max_tries + 1):
                try:
                    return func(*a, **k)
                except exceptions as e:
                    print(f"  attempt {attempt} failed: {e}")
                    if attempt == max_tries:
                        raise
        return wrapper
    return deco

calls = {"n": 0}

@retry(max_tries=3, exceptions=(ConnectionError,))
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("network down")
    return "ok on call %d" % calls["n"]
print(flaky())

@retry(max_tries=2)
def always_bad():
    raise ValueError("nope")
try:
    always_bad()
except ValueError as e:
    print("gave up:", e)

  attempt 1 failed: network down
  attempt 2 failed: network down
ok on call 3
  attempt 1 failed: nope
  attempt 2 failed: nope
gave up: nope


In [ ]:
import functools, time

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)
print(fib(80), fib.cache_info())

def fib_slow(n):
    return n if n < 2 else fib_slow(n-1) + fib_slow(n-2)
t0 = time.perf_counter(); fib_slow(27); slow_t = time.perf_counter() - t0
print("uncached fib(27) slower than 5ms:", slow_t > 0.005)

@functools.cache
def sq(x):
    print("computing", x)
    return x * x
print(sq(4), sq(4))

def tag(name):
    def deco(f):
        @functools.wraps(f)
        def w(*a, **k):
            return f"<{name}>{f(*a, **k)}</{name}>"
        return w
    return deco

@tag("b")
@tag("i")
def text():
    return "hi"
print(text())

def add_repr(cls):
    def __repr__(self):
        items = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({items})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
print(Point(1, 2))

registry = {}
def register(f):
    registry[f.__name__] = f
    return f

@register
def clean(): return "cleaning"
@register
def load(): return "loading"
print(sorted(registry), registry["load"]())

23416728348467685 CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)
uncached fib(27) slower than 5ms: True
computing 4
16 16
<b><i>hi</i></b>
Point(x=1, y=2)
['clean', 'load'] loading


## 7.4 Context Managers

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        print("enter")
        return self
    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.t0
        print("exit, exception:", exc_type.__name__ if exc_type else None)
        return False

with Timer() as t:
    time.sleep(0.01)
print(t.elapsed > 0.005)

try:
    with Timer():
        1/0
except ZeroDivisionError:
    print("error still propagates")

enter
exit, exception: None
True
enter
exit, exception: ZeroDivisionError
error still propagates


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import contextlib, os

class Suppress:
    def __enter__(self):
        return self
    def __exit__(self, et, e, tb):
        return et is KeyError

with Suppress():
    {}["x"]
print("KeyError swallowed")

with contextlib.suppress(FileNotFoundError):
    os.remove("/tmp/definitely_not_here_123")
print("suppress ok")

KeyError swallowed
suppress ok


In [ ]:
import contextlib, os, tempfile, io

@contextlib.contextmanager
def chdir_temp():
    old = os.getcwd()
    d = tempfile.mkdtemp()
    os.chdir(d)
    try:
        yield d
    finally:
        os.chdir(old)
        os.rmdir(d)
        print("restored")

with chdir_temp() as d:
    print(os.path.samefile(os.getcwd(), d))

@contextlib.contextmanager
def opened(name):
    print("open", name)
    try:
        yield name
    finally:
        print("close", name)

with opened("a") as a, opened("b") as b:
    print("inside", a, b)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print("captured")
print(repr(buf.getvalue()))

with contextlib.ExitStack() as st:
    fs = [st.enter_context(opened(n)) for n in "xy"]
    print("both open", fs)

True
restored
open a
open b
inside a b
close b
close a
'captured\n'
open x
open y
both open ['x', 'y']
close y
close x


In [ ]:
f = open("data.txt", encoding="utf-8")
total = 1 / 0        # error: the next line never runs
f.close()

## 7.5 Two-Way Generators, Parametrised Decorators and Stacked Cleanup

Sections 7.2 to 7.4 used each feature in its simplest form. This section covers the forms that appear in real code: generators that receive values, decorators that take settings, and context managers that you assemble at run time.

In [ ]:
import inspect
def running_avg():
    total = 0.0
    count = 0
    avg = None
    while True:
        x = yield avg
        total += x
        count += 1
        avg = total / count
g = running_avg()
print(inspect.getgeneratorstate(g))
next(g)
print(inspect.getgeneratorstate(g))
print(g.send(10), g.send(20), g.send(60))
g.close()
print(inspect.getgeneratorstate(g))

GEN_CREATED
GEN_SUSPENDED
10.0 15.0 30.0
GEN_CLOSED


In [ ]:
def inner():
    yield 1
    yield 2
    return "done"
def outer():
    result = yield from inner()
    print("inner returned", result)
    yield 3
print(list(outer()))
def cleanup_demo():
    try:
        yield 1
        yield 2
    finally:
        print("cleanup ran")
g = cleanup_demo()
print(next(g))
g.close()

inner returned done
[1, 2, 3]
1
cleanup ran


In [ ]:
import functools
def retry(times=3, exceptions=(ConnectionError,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator
calls = {"n": 0}
@retry(times=3)
def flaky(x):
    """Return x after two failures."""
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("down")
    return x * 2
print(flaky(21), flaky.__name__, flaky.__doc__)
@retry(times=2, exceptions=(ValueError,))
def bad():
    raise ValueError("nope")
try:
    bad()
except ValueError as e:
    print("gave up:", e)

attempt 1 failed: down
attempt 2 failed: down
42 flaky Return x after two failures.
attempt 1 failed: nope
attempt 2 failed: nope
gave up: nope


In [ ]:
from contextlib import contextmanager, ExitStack
@contextmanager
def step(name):
    print("enter", name)
    try:
        yield name
    except ZeroDivisionError:
        print("handled in", name)
    finally:
        print("exit", name)
with step("A") as a:
    1 / 0
    print("not reached")
with ExitStack() as stack:
    names = [stack.enter_context(step(n)) for n in ("x", "y", "z")]
    print(names)

enter A
handled in A
exit A
enter x
enter y
enter z
['x', 'y', 'z']
exit z
exit y
exit x


In [ ]:
from itertools import islice
def evens():
    n = 0
    while True:
        yield n
        n += 2
print(list(islice(evens(), 5)))

[0, 2, 4, 6, 8]


In [ ]:
def loud(func):
    def wrapper(*a, **k):
        return func(*a, **k).upper()
    return wrapper
@loud
def greet(name):
    "Say hello."
    return "hi " + name
print(greet("ana"), greet.__name__, greet.__doc__)

HI ANA wrapper None


In [ ]:
import time
from contextlib import contextmanager
@contextmanager
def timer(label, sink):
    start = time.perf_counter()
    try:
        yield
    finally:
        sink[label] = time.perf_counter() - start
t = {}
try:
    with timer("job", t):
        raise RuntimeError("boom")
except RuntimeError:
    pass
print("job" in t, t["job"] >= 0)

True True


## Chapter recap

## Practice set

Problem 7.P.1 What is `next(iter([5, 6]))`?

In [ ]:
g = (x * 2 for x in [1, 2, 3])
print(list(g))
print(list(g))

In [ ]:
def f():
    yield 1
    yield 2
print(f())

In [ ]:
def evens(limit):
    for n in range(limit):
        if n % 2 == 0:
            yield n

In [ ]:
def shout(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        func(*args, **kwargs)
    return wrapper

@shout
def add(a, b):
    return a + b
print(add(2, 3))

In [ ]:
@tag("b")
@tag("i")
def text(): return "hi"

In [ ]:
import functools, logging, time
log = logging.getLogger(__name__)

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            log.info("%s took %.1f ms", func.__name__, (time.perf_counter() - t0) * 1000)
    return wrapper

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch07.html